## Multi-dimensional

There are two "layers" (so, a 2 dimensional graph) of edges on the same set of nodes (the compute tasks). The first layer represents the dependencies between compute tasks, the second layer represents the devices' topological orderings, i.e., the schedule.

A formal representation could be:

$G = (V, E_D, E_S)$, where $V$ is the set of nodes, $E_D$ is the set of hyperedges in the dependency graph and $E_S$ is the set of directed edges in the scheduling graph. We use $G_D = (V, E_D)$ as the dependency hypergraph, and $G_S = (V, E_S)$ as the scheduling forest.

## Layer 1: Dependency Hypergraph

The dependency layer is an acyclic directed hypergraph, meaning that edges can connect any amount of nodes, but are directed. Edges always are directed *from* exactly one node *to* one or more nodes. The "*to*" nodes are directly dependent on the "*from*" node. The data movement 

Hyperedges allow to simply realize that the same data is transferred by this edge, as opposed to multiple distinct edges for each target node.

The total set of all possible hyperedges is $\mathbb{E}_D = \left\{\left(v_{in}, s\right) | v_{in} \in V, s \in \mathcal{P}(V) \setminus \emptyset \right\}$. The set of edges of $G_D$ is then a subset: $E_D \subset \mathbb{E}_D$.

A node $v_1$ is called *directly dependent* on a node $v_2$ if there exists an edge $e = (v_2, \{v_1, \dots \})$ in $E_D$.

A node $v_1$ is *dependent* on a node $v_2$ if there exists a path from $v_2$ to $v_1$ in $E_D$ (*formalize*).

A hypergraph is called acyclic when no non-empty path from any node to itself exists (*formalize*). $G_D$ is acyclic.

A hypergraph is called connected when (?) *formalize*. $G_D$ is connected. $G_D$ must be acyclic and fully connected. 

$E_D$ can not contain multiple edges with the same *from* node (*formalize*).

The only nodes that have no dependencies are *entry nodes*, the only nodes that have no dependents are *exit nodes*. In a non-empty graph, both entry nodes and exit nodes are guaranteed to exist by the other conditions.

## Layer 2: Scheduling Forest

The schedule layer is a forest, with one tree per device. This graph is not a hyper graph, each directed edge connects a node to the next node scheduled to the same device. Since there can only ever be one next node, this makes the trees unary, similar to a linked list.

$G_S$ must be acyclic. It must be a forest of at most $N$ unary trees (basically strings, or linked lists), where $N$ is the number of devices the graph is scheduled on. A device may remain unused and the initial state might be a topological ordering on a single device, so we can't require exactly $N$ unary trees.

A node $v_1$ is called *directly subsequent* to a node $v_2$ if there exists an edge $e = (v_2, v_1)$ in $E_S$ (*formalize*).

A node $v_1$ is called subsequent to a node $v_2$ if there exists a path from $v_2$ to $v_1$ in $E_S$ (*formalize*).

If a node $v_1$ is subsequent to a node $v_2$, then $v_2$ must not be dependent on $v_1$. This ensures topological ordering of the schedule.



In [7]:
using UUIDs

function data end
function compute_effort end

abstract type Task end

# the CDAG holds a full map of uuid -> node for indexing because julia doesn't like complex interref stuff
mutable struct Node
    id::UUID

    task::Task

    # **the hypergraph edges**
    # dependencies are the nodes this node depends on for inputs, the associated integers define their argument order
    dependencies::Vector{Tuple{UUID, Int}}

    # dependents are the nodes dependent on this node
    dependents::Vector{UUID}

    # **the scheduler edges**
    # antecedent is the node scheduled after this one on the same device. a special value can be used to define "none", like UUID(0)
    antecedent::UUID

    # precedent is the node scheduled before this one on the same device. a special value can be used to define "none", like UUID(0)
    precedent::UUID

    # **node scheduler metrics**
    # largest sum of path from entry node to this node
    # strongly correlates with earliest start time
    t_level::Float64

    # largest sum of path from this node to exit node
    # bounded by critical path
    b_level::Float64
end

# the t-level of a node may (or may not) change whenever a node it is dependent on changes its t_level
# the b-level of a node may (or may not) change whenever a node that depends on it changes its b_level

In [5]:
struct Hyperedge
    # id of the edge
    id::UUID

    # uuid of the "in" node
    dependency::UUID

    # uuids of the out nodes, and the index of the dependency for each of them
    dependents::Vector{Tuple{UUID, Int}}
end

## Operations

Requirement: Must be *total*, i.e., all valid equivalent states of the DAG must be reachable from every state. This includes all DAGs that are known to be equivalent under application of known algebraic properties.

In the space of sets of operations that can do that, find something at an appropriate level of abstraction, that is a good base for optimization.
Question: Should the set of operations be *minimal*? That is, if any of them are missing, it's no longer *total*?

All operations should aim to have relatively small changes in b-levels and t-levels of other nodes. This way, it might be enough to only update the whole graph's b-/t-levels periodically every so often, instead of after every change.

### Node Duplication

Duplicate a node. The newly generated nodes get inserted on other devices' schedules. The new nodes inherit the dependencies of the original node, and each of the nodes gets one of the originals dependent nodes.

Question: Where exactly should the new nodes be inserted in the schedules?
Options:
- Try earliest/latest approximate time point, but difficult to determine where that is exactly. Base on t-level, b-level of dependent nodes/dependency nodes.
- Try same time point as the original node on the other device.

### Node Reduction

Inverse of the node duplication, two (or more) nodes with equivalent type and dependencies merge into one with both of the nodes' dependents.
One of the nodes is chosen as the main one, which stays, while the others are deleted.

### Term Rewriting

According to some arithmetic property, a subgraph in the dependency graph is rewritten as a different but equivalent subgraph, possibly with different node types, connections, and inputs.

### Rescheduling

A node is moved from one device to another device.

Similar question as with node duplication, where to insert in the timeline?

### Reordering

A node is moved forward or backward in the schedule of the same device some number of spots.

### Node Vectorization

Multiple nodes with the same type but different inputs are collected into one "vectorized" node, that applies the same function to all the inputs.

The original nodes don't necessarily have to be on the same device, but the resulting node obviously can only run on one device.